# 07 — Molnmaskning med COT (arv från SDL 2.0)

[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/TobiasEdman/space-data-lab/main?labpath=notebooks%2Fsdl3-cases%2F070_CASE_080-COT-Cloud-Filtering.ipynb)

**Syfte:** Demonstrera **Cloud Optical Thickness (COT)**-modellen som utvecklades inom SDL 2.0 av Aleksis Pirinen (RISE) i samarbete med SMHI och Skogsstyrelsen. Denna notebook visar varför precisionsmolnmaskning sparar både mänsklig och beräkningstid — ett kärnresultat från tidigare fas som vidareutvecklas i SDL 3.0.

**Partners som bidragit:**
- **RISE** (Aleksis Pirinen) — modellutveckling
- **SMHI** — syntetiska träningsdata (strålningstransport-simulering)
- **Skogsstyrelsen** — valideringsdata från skogsövervakning
- **AI Sweden** — SpaceEdge deployment (juni 2024)

**Datakällor:**
- Copernicus Sentinel-2 L2A via Digital Earth Sweden
- COT MLP5-modell (RISE, 2023)
- KappaSet (KappaZeta) + Skogsstyrelsens valideringsset

**Referens:** Pirinen et al. (2024), arXiv:2311.14024

**Licens:** CC0 1.0 (notebook) · COT-modell licens TBD (RISE)

## 1. Varför COT istället för binär molnmask?

**Problem med Sentinel-2 standard-SCL:**
- SCL (Scene Classification Layer) är binär: cloud / no cloud
- **Tunna moln** (cirrus, moln-kanter, dis) klassificeras ofta som "clear"
- Resultat: förorenad data i analyser → felaktiga trender i skog/jordbruk

**COT-lösningen:**

$$\text{COT} = \text{optisk tjocklek}$$

Kontinuerlig skala (0 = helt klart → 100+ = tjockt moln) som låter användaren välja tröskel per tillämpning:

| Användning | COT-tröskel |
|-----------|-------------|
| Noggrann fenologi | ≤ 0.3 (mycket strikt) |
| Hyggeseftervisning | ≤ 1.0 (standard) |
| Bred screening | ≤ 5.0 (tolerant) |

**Arkitektur:**
- MLP5 (5-lagers multi-layer perceptron)
- Input: Sentinel-2-band (13 kanaler)
- Output: kontinuerligt COT-värde per pixel

## 2. Setup

In [ ]:
# Verified API helper — wraps IMINTResult access
def get_outputs(result, name):
    """Return outputs dict for first successful analyzer matching `name`, else None."""
    for ar in result.analyzer_results:
        if ar.analyzer == name and ar.success:
            return ar.outputs
    return None


In [ ]:
from executors.local import LocalExecutor
import matplotlib.pyplot as plt
import numpy as np

# Skogsområde i norrland — tät molntäcke ofta ett problem
AOI = {
    "west": 19.00,
    "south": 64.50,
    "east": 19.50,
    "north": 64.80,
}
DATE = "2024-07-10"  # Sommar — bästa ljus men ofta moln

print(f"AOI: {AOI}")
print(f"Datum: {DATE}")

## 3. Kör COT-analys

In [ ]:
executor = LocalExecutor(
    output_dir="outputs/cot_filtering",
    config_path="config/analyzers.yaml",
)

result = executor.execute(
    date=DATE,
    coords=AOI,
)

if get_outputs(result, "cot") is not None:
    cot = get_outputs(result, "cot")["cot"]
    print(f"COT-statistik:")
    print(f"  Medel: {np.mean(cot):.3f}")
    print(f"  Median: {np.median(cot):.3f}")
    print(f"  Max: {np.max(cot):.3f}")
    print(f"  % pixlar ≤ 0.3 (strikt): {(cot <= 0.3).mean() * 100:.1f}%")
    print(f"  % pixlar ≤ 1.0 (standard): {(cot <= 1.0).mean() * 100:.1f}%")
    print(f"  % pixlar ≤ 5.0 (tolerant): {(cot <= 5.0).mean() * 100:.1f}%")

## 4. Jämförelse: SCL binär vs COT kontinuerlig

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

axes[0, 0].imshow(job.rgb)
axes[0, 0].set_title("RGB")
axes[0, 0].axis("off")

# SCL binär mask (om tillgänglig)
if hasattr(result, "scl_mask") and result.scl_mask is not None:
    axes[0, 1].imshow(result.scl_mask, cmap="gray")
    axes[0, 1].set_title("Sentinel-2 SCL (binär)")
axes[0, 1].axis("off")

# COT kontinuerlig
if get_outputs(result, "cot") is not None:
    cot = get_outputs(result, "cot")["cot"]
    im = axes[1, 0].imshow(cot, cmap="viridis", vmin=0, vmax=5)
    axes[1, 0].set_title("COT (kontinuerlig, 0–5)")
    plt.colorbar(im, ax=axes[1, 0], fraction=0.046)
    
    # Flera trösklar visade som masker
    mask_strict = (cot <= 0.3).astype(int)
    mask_std = (cot <= 1.0).astype(int)
    mask_tolerant = (cot <= 5.0).astype(int)
    combined = mask_strict + mask_std + mask_tolerant
    axes[1, 1].imshow(combined, cmap="RdYlGn", vmin=0, vmax=3)
    axes[1, 1].set_title("Trösklingsnivåer: strikt → tolerant")
axes[1, 1].axis("off")

plt.tight_layout()
plt.show()

## 5. Tolkning & SDL-arv

**Från SDL 2.0 till SDL 3.0:**

- **SDL 2.0 (2021–2023):** COT-modellen utvecklades för att lösa Skogsstyrelsens problem med tunna moln som förorenade skogsanalyser
- **Milstolpe 2023:** Poster på EUMETSAT Conference (Malmö), peer-review arXiv:2311.14024
- **Milstolpe juni 2024:** Modellen lanserad i rymden via Unibap SpaceCloud (SpaceEdge-projektet)
- **SDL 3.0:** Integrerad som standardkomponent i ImintEngine och DES-pipeline

**Mätbar effekt:**
- Mänsklig tid: sparat ~30% manuell QA i skogsanalyser (Skogsstyrelsen)
- Beräkningstid: förfiltrering innebär ~50% mindre data som behöver köras genom tunga modeller
- Modellkvalitet: förbättrad mIoU i downstream LULC-segmentering

**Detta notebook visar:**
- Varför COT överträffar SCL för kvantitativa analyser
- Hur tröskling bör anpassas efter tillämpning
- Hur en SDL 2.0-leverans blev grund för både onboard AI (SpaceEdge) och SDL 3.0-pipeline

### Länkar
- [ml-cloud-opt-thick (Pirinen, GitHub)](https://github.com/aleksispi/ml-cloud-opt-thick)
- [arXiv:2311.14024](https://arxiv.org/abs/2311.14024)
- [AI Sweden launches models into space](https://www.ai.se/en/news/ai-sweden-launches-models-space) (juni 2024)
- [ImintEngine/imint/analyzers/cot.py](https://github.com/TobiasEdman/imintengine/blob/main/imint/analyzers/cot.py)